# Load Packages

In [ ]:
# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join
import joblib
import sys
import torch
sys.path.append("../../")

from src.configs.crc_config import data_name
from src.file_manager.filepath import FilePath
from src.training.misc import get_pos_weight

batch_size = 32
eval_batch_size = 128
seed = 2024

seed=2024
fp = FilePath(data_name=data_name, seed=seed)
fp_data_file = join(fp.get_preprocessed_folder(), "raw_data", "colorectalcancers_schs_pgs.csv")
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

# Load Data

In [ ]:
df = pd.read_csv(fp_data_file, index_col=0)
df

In [ ]:
df.columns

# Data Exploration

In [ ]:
def get_col_types(df, categorical_cols, target_col, exclude_col=None):
    continuous_cols = [col for col in df.columns if col not in categorical_cols + [target_col]]
    if exclude_col:
        continuous_cols.remove(exclude_col)
    print(f"{len(categorical_cols)} Categorical Columns:", categorical_cols)
    print(f"{len(continuous_cols)} Continuous Columns:", continuous_cols)
    print("Target Column:", target_col)
    return continuous_cols
categorical_cols = [
    "Sex (1=Male, 2=Female)", "alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)", 
    "smoke_never(0)_ex(1)_current(2)", "Prevalent_diabetes"
]

target_col = "colorectal cancer"
exclude_col = 'Follow-up time'
continuous_cols = get_col_types(df, categorical_cols, target_col, exclude_col)

In [ ]:
df[continuous_cols].describe()

In [ ]:
def show_continuous_features(df, continuous_cols, clean_col_names=None):
    from math import ceil
    nrows = 2
    ncols = ceil(len(continuous_cols)/2)
    size = 2
    fig, axes = plt.subplots(nrows, ncols, dpi=300, figsize=(size*ncols, size*nrows))
    axes = axes.flatten()
    axes[0].set_ylabel("Count")
    for i, col in enumerate(continuous_cols):
        df[col].hist(bins=10, ax=axes[i])
        axes[i].set_xlabel(clean_col_names[i] if clean_col_names else col) 
    for j in range(i+1, ncols*nrows):
        axes[j].set_axis_off()
    plt.tight_layout()
    
show_continuous_features(
    df,continuous_cols, 
)      

# Data Processing

## Drop NA

In [ ]:
def drop_na(df):
    df = df.copy()
    display(df.isna().sum())
    print(f"DataFrame Shape Before Dropping NA: {df.shape}")
    df = df.dropna()
    print(f"DataFrame Shape Before Dropping NA: {df.shape}")
    return df
df = drop_na(df)
display(df)

In [ ]:
for col in categorical_cols + [target_col]:
    print(df[col].value_counts())

## Data Split

In [ ]:
# Split data
def split_data_stratified(df, target_col, val_prop, test_prop, seed):
    np.random.seed(seed)
    df = df.copy()
    data_size = len(df)
    train_prop = 1 - val_prop - test_prop
    train_dfs, val_dfs, test_dfs = [], [], []
    # From each class, choose the same proportion of val and test
    for target, group_df in df.groupby(target_col):
        group_size = len(group_df)
        indices = np.array([i for i in range(group_size)])
        np.random.shuffle(indices)
        train_size, val_size = round(train_prop * group_size), round(val_prop * group_size)
        train_indices, val_indices, test_indices = (
            indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:])
        train_dfs.append(group_df.iloc[train_indices])
        val_dfs.append(group_df.iloc[val_indices])
        test_dfs.append(group_df.iloc[test_indices])
    return dict(
        train_df=pd.concat(train_dfs), val_df=pd.concat(val_dfs), test_df=pd.concat(test_dfs)
    )

split_dict = split_data_stratified(df, target_col, val_prop=0.1, test_prop=0.1, seed=seed)
for label, cur_df in split_dict.items():
    print(label,":")
    print("- Total Number of Samples:", len(cur_df))
    print("- Class Distribution:", cur_df[target_col].value_counts()/len(cur_df))

## Binarisation + One-Hot Encoding

In [ ]:
def one_hot_encode_data(train_df, val_df, test_df, categorical_cols):
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    
    # Binarise "Sex" col
    prev_col_name = "Sex (1=Male, 2=Female)"
    new_col_name = "Sex (0=Male, 1=Female)"
    train_df[new_col_name] = train_df[prev_col_name] - 1
    val_df[new_col_name] = val_df[prev_col_name] - 1
    test_df[new_col_name] = test_df[prev_col_name] - 1

    # One-Hot Encode "smoke_never" col
    prev_col_name =  'smoke_never(0)_ex(1)_current(2)'
    new_col_names = ["smoke_ex(1)", "smoke_current(2)"]
    from sklearn.preprocessing import OneHotEncoder
    encoder = OneHotEncoder(drop="first")
    train_df[new_col_names] = encoder.fit_transform(train_df[[prev_col_name]]).toarray()
    val_df[new_col_names] = encoder.transform(val_df[[prev_col_name]]).toarray()
    test_df[new_col_names] = encoder.transform(test_df[[prev_col_name]]).toarray()
    
    train_df = train_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    val_df = val_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    test_df = test_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    ), encoder

split_dict_encoded, encoder = one_hot_encode_data(
    **split_dict, categorical_cols=categorical_cols)

for label, cur_df in split_dict_encoded.items():
    print(label,":")
    display(cur_df)

In [ ]:
# Save encoder
joblib.dump(encoder, fp_encoder_file) 

In [ ]:
# Check that categorical variables correspond to pre-processed distributions
categorical_cols = [
 "Sex (0=Male, 1=Female)", "alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)", 
    "smoke_ex(1)", "smoke_current(2)", "Prevalent_diabetes"
]
_ = get_col_types(split_dict_encoded["train_df"], categorical_cols, target_col, exclude_col)

In [ ]:
testing_df = pd.concat(list(split_dict_encoded.values()))
for col in categorical_cols + [target_col]:
    print(testing_df[col].value_counts())

## Data Normalisation

In [ ]:
def normalise_data(train_df, val_df, test_df, continuous_cols):
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    train_df[continuous_cols] = scaler.fit_transform(train_df[continuous_cols])
    val_df[continuous_cols] = scaler.transform(val_df[continuous_cols])
    test_df[continuous_cols] = scaler.transform(test_df[continuous_cols])
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    ), scaler

split_dict_scaled, scaler = normalise_data(**split_dict_encoded, continuous_cols=continuous_cols)
for label, cur_df in split_dict_scaled.items():
    print(label,":")
    display(cur_df[continuous_cols].describe())

In [ ]:
# Save scaler
joblib.dump(scaler, fp_scaler_file) 

In [ ]:
# Check that continuous variables have been scaled
for label, cur_df in split_dict_scaled.items():
    print(label,":")
    display(cur_df[continuous_cols].describe())

In [ ]:
joblib.dump(split_dict_scaled, fp_split_dict_file)

## Oversample Data

In [ ]:
def oversample_data(train_df, val_df, test_df, target_col):
    from imblearn.over_sampling import SMOTE
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    oversample = SMOTE(random_state=seed)
    feature_cols = [col for col in train_df.columns if col != target_col]
    X, y = oversample.fit_resample(train_df[feature_cols], train_df[target_col])
    train_df = pd.DataFrame(X, columns=feature_cols)
    train_df[target_col] = y
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    )

split_dict_oversampled = oversample_data(
    **split_dict_scaled, target_col=target_col)
split_dict_oversampled["train_df"][target_col].value_counts()

In [ ]:
feat_cols_w_pc = continuous_cols + categorical_cols
print(f"{len(feat_cols_w_pc)} Feature Columns With PCs: {feat_cols_w_pc}")

In [ ]:
selected_feat_cols = ['SBP', 'DBP', 'DASH', 'BMI', 'aMED', 'Leisure screen time', 'Age_interview']
print(f"{len(selected_feat_cols)} Selected Feature Columns: {selected_feat_cols}")

In [ ]:
for split in ["train_df", "val_df", "test_df"]:
    print(split_dict_oversampled[split][target_col].value_counts())

In [ ]:
for split in ["train_df", "val_df", "test_df"]:
    print(f"Total: {len(split_dict_scaled[split])}")
    print(split_dict_scaled[split][target_col].value_counts())

In [ ]:
16551+2069+2069

In [ ]:
joblib.dump(split_dict_oversampled, fp_split_dict_oversampled_file)

## Class Weights

In [ ]:
pos_weight = get_pos_weight(split_dict_scaled, target_col)
pos_weight